# DataReader Class  

Todo está hecho de cara a trabajar con el Dataset CUB

In [1]:
import scipy.io 
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.svm import SVC 
from sklearn.feature_selection import RFE
import pathlib
import typing
from typing import List, Tuple, Dict, Any, Callable, Type
import numpy.typing as npt


from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator
from collections import Counter


Scaler = Type[BaseEstimator]
from datareader import *

In [2]:
path      = pathlib.Path('data/')
path_cub  = path / 'CUB/'

SHOW = True

# Lectura en frío de los datos

In [7]:
SHOW = False 

# Set up the path to the data
path      = pathlib.Path('data/')

path_cub  = path / 'CUB/'
path_awa2 = path / 'AWA2/'
path_FLO  = path / 'FLO/'
path_sun  = path / 'SUN/'


# Load the data
matcontent, att_splits = load_data_from_path(path_cub, show = False )

"""
Comentar que en los datos de este datasets viene una partición establecida para train y val. Sin embargo, la partición de 'val' no contempla clases Unseen. Por tanto no la vamos usar.
Crearemos una partición de validación para clases Seen y Unseen a partir de la partición de test. 
Como conjunto de entrenamiento usaremos el que viene dado por los índices de la variable 'trainval_loc' en att_splits. 
Los índices de las varaibles 'train_loc' y 'val_loc' no los vamos a usar.
"""
# Get the data from att_splits dictionary
allclasses_names            = att_splits['allclasses_names']
original_att                = att_splits['original_att'].T.astype('float')  # Attributos originales
attribute                   = att_splits['att'].T.astype('float')           # Atributos que no se de donde salen, pero son los que usan <=== IMPORTANTE
# Indixes of the instances in the train, test and validation sets
test_unseen_loc             = att_splits['test_unseen_loc'].squeeze() - 1
test_seen_loc               = att_splits['test_seen_loc'].squeeze() - 1
trainval_loc                = att_splits['trainval_loc'].squeeze() - 1      # En caso de no usar validación, aquí estara todo el entrenamiento
train_loc                   = att_splits['train_loc'].squeeze() - 1         # Esto es si usamos validacion, ver comentario arriba
val_loc                     = att_splits['val_loc'].squeeze() -1            # Esto es si usamos validacion, ver comentario arriba
# Get the dta from matcontent dictionary
feature =  matcontent['features'].T                       # Feature Matrix (Number of instances x Number of features)
labels  =  matcontent['labels'].astype(int).squeeze() - 1 # Ponemos que sea entero, eliminamos la dimension extra y con el '-1' movemos el rango de [1,200] a [0,199]



# Obtaining train and test data
train_feature         = feature[trainval_loc]
test_seen_feature     = feature[test_seen_loc]
test_unseen_feature   = feature[test_unseen_loc]

# Obtaining train and test labels
train_label          = labels[trainval_loc]
test_unseen_label    = labels[test_unseen_loc]
test_seen_label      = labels[test_seen_loc]



In [ ]:




##########################################################################################################################################
############################################### PARTICIÓN EN VALIDACIÓN Y TEST  ##########################################################                    
##########################################################################################################################################


##############################################
# Partición TEST_SEEN & VAL_SEEN
##############################################
""" 
Hacemos una partición del conjunto de test_seen en dos partes, una para validación y otra para test.
Usamos la función train_test_split de sklearn para hacer la partición de manera estratificada. 
"""
# Particionamos el conjunto de test_seen en dos partes, una para validación y otra para test
seed = 42 
percentage_test_seen = 0.6
np.random.seed(seed)
val_seen_feature, test_seen_feature, val_seen_label, test_seen_label = train_test_split(test_seen_feature, test_seen_label, test_size=percentage_test_seen, random_state=seed,stratify=test_seen_label)

# Imprimo información sobre tamaños
if SHOW:
    print(f"val_seeen_feature: {val_seen_feature.shape}")
    print(f"test_seen_feature: {test_seen_feature.shape}")
    print(f"val_seen_label: {val_seen_label.shape}")
    print(f"test_seen_label: {test_seen_label.shape}")
    print(f"Counter(test_seen_label_feature): {Counter(test_seen_label)}")
    print(f"Counter(val_seen_label_feature): {Counter(val_seen_label)}")



##############################################
# Partición TEST_UNSEEN & VAL_UNSEEN
##############################################
"""
Hacemos una partición del conjunto de test_unseen en dos partes, una para validación y otra para test.
El array test_unseen_label contiene las etiquetas de manera contigua, no está desordenado. 
"""
# Calculamos el numero total de clases no vistas y seleccionamos el porcentage para validacion 
total_classes = np.unique(test_unseen_label).shape[0]
percentage_test_unseen = 0.3

# Cogemos qué clases tendrá el conjunto de validación y el de test (los elementos de los vectores son las clases)
val_unseen_classes  = np.unique(test_unseen_label)[:int(total_classes*(1- percentage_test_unseen))]
test_unseen_classes = np.unique(test_unseen_label)[int(total_classes*(1- percentage_test_unseen)):]
# Calculo el conjunto de etiquetas y entrenamiento de validación para unseen 
# Recorro el vector de etiquetas del conjunto de test original y la añado si está dentro del conjunto de clases que he seleccionado para validacion 
val_unseen_label = np.array([i for i in test_unseen_label if i in val_unseen_classes])
# Recorro todos los elementos del conjunto de entrenamiento y añado aquellos cuya etiqueta asociada esté dentro del conjunto de validacion que he seleccinado
val_unseen_feature = np.array([test_unseen_feature[i] for i in range(test_unseen_feature.shape[0]) if test_unseen_label[i] in val_unseen_classes])

# Calculo el conjunto de etiquetas y entrenamiento de test para unseen
# Recorro el vector de etiquetas del conjunto de test original y la añado si está dentro del conjunto de clases que he seleccionado para test
test_unseen_label_aux = np.array([ i for i in test_unseen_label if i in test_unseen_classes])
# Recorro todos los elementos del conjunto de entrenamiento y añado aquellos cuya etiqueta asociada está en el conjunto de test que he seleccionado
test_unseen_feature_aux = np.array([test_unseen_feature[i] for i in range(test_unseen_feature.shape[0]) if test_unseen_label[i] in test_unseen_classes])
test_unseen_label    = test_unseen_label_aux 
test_unseen_feature = test_unseen_feature_aux



if SHOW:
    print(f"Unseen Clases: {np.unique(test_unseen_label)}")
    print(f"Size: {total_classes}")
    print(f"val_unseen_classes: {np.unique(val_unseen_classes).shape}") 
    print(f"test_unseen_classes: {np.unique(test_unseen_classes).shape}")
    # Imprimo los tamaños 
    print(f"val_unseen_label: {val_unseen_label.shape}")
    print(f"val_unseen_feature: {val_unseen_feature.shape}")
    print(f"test_unseen_label: {test_unseen_label.shape}")
    print(f"test_unseen_feature: {test_unseen_feature.shape}")


assert all([i in val_unseen_classes for i in val_unseen_label]), "Error: There are labels in val_unseen_label_split that are not in val_unseen_classes"
assert all([i in test_unseen_classes for i in test_unseen_label]), "Error: There are labels in test_unseen_label_split that are not in test_unseen_classes"   



In [ ]:

##########################################################################################################################################
####################################################### PREPROCESSING 1 ##################################################################
##########################################################################################################################################
preprocessing_opt   = True  # Escalado de los atributos y las características
orig_attribute      = False # Attributos originales u otros
normalize_att       = True
scaler_str          = 'Standard'     # ['Standard','MinMax']

# Choosing the attributes we want to use 
attribute = original_att if orig_attribute else attribute

# Normalizamos los atributos dividiendo por la norma de cada vetor de atributos
if normalize_att:
    attribute    = l2_normalization(attribute)

print(attribute.shape)
# Applying the scaler to the data if necessary
if preprocessing_opt:
    scaler = get_scaler(scaler_str)
    train_feature, val_seen_feature, val_unseen_feature, test_seen_feature, test_unseen_feature = scaler_data(scaler,train_feature,val_seen_feature,val_unseen_feature,test_seen_feature,test_unseen_feature)
    # Scaler and transfer attributes
    scaler = get_scaler(scaler_str)
    attribute = scaler_data(scaler, attribute)


# Espacios semanticos asociados a los conjuntos de X_train, test_seen y test_unseeen. 
# Estos SS serán matrices con tantas filas como ejemplos y columnas como atributos
# Los vectores de atributios (columnas) se repiten para aquellas instancias que tengan la misma clase
S_train          = gen_ss_from_data(train_label,attribute)
S_test_seen      = gen_ss_from_data(test_seen_label,attribute)
S_test_unseen    = gen_ss_from_data(test_unseen_label,attribute)
S_val_seen       = gen_ss_from_data(val_seen_label,attribute)
S_val_unseen     = gen_ss_from_data(val_unseen_label,attribute)
semantic_gt      = gen_ss_from_data(np.unique(test_seen_label),attribute)
S_gt_training    = gen_ss_from_data(np.unique(train_label),attribute)
S_gt_val_seen    = gen_ss_from_data(np.unique(val_seen_label),attribute)
S_gt_val_unseen  = gen_ss_from_data(np.unique(val_unseen_label),attribute)
S_gt_test_seen   = gen_ss_from_data(np.unique(test_seen_label),attribute)
S_gt_test_unseen = gen_ss_from_data(np.unique(test_unseen_label),attribute)


if SHOW:
    print(f"S_train: {S_train.shape}")
    print(f"S_test_seen: {S_test_seen.shape}")
    print(f"S_test_unseen: {S_test_unseen.shape}")
    print(f"S_val_seen: {S_val_seen.shape}")
    print(f"S_val_unseen: {S_val_unseen.shape}")
    print(f"semantic_gt: {semantic_gt.shape}")
    print(f"S_gt_training: {S_gt_training.shape}")
    print(f"S_gt_val_seen: {S_gt_val_seen.shape}")
    print(f"S_gt_val_unseen: {S_gt_val_unseen.shape}")
    print(f"S_gt_test_seen: {S_gt_test_seen.shape}")
    print(f"S_gt_test_unseen: {S_gt_test_unseen.shape}")
    




##########################################################################################################################################
###################################################### Creating datasets #################################################################
##########################################################################################################################################
training_data = Dataset(train_feature, train_label, S_train, mode = 'train')

test_seen     = Dataset(test_seen_feature, test_seen_label, S_test_seen, mode = 'test')
test_unseen   = Dataset(test_unseen_feature, test_unseen_label, S_test_unseen, mode = 'test')

val_seen      = Dataset(val_seen_feature, val_seen_label, S_val_seen, mode = 'val')
val_unseen    = Dataset(val_unseen_feature, val_unseen_label, S_val_unseen, mode = 'val')

test_data     = StackedDataset({'test_seen':test_seen, 'test_unseen':test_unseen})
val_data      = StackedDataset({'val_seen':val_seen, 'val_unseen':val_unseen})

# Particionado de los datos en Training, Val y Test
- División de Training: 
    - Training (Seen)
    - Validación (Seen / Unseen)

Nos olvidamos del conjunto de test. Vamos primero a estudiar cuantas etiquetas hay en el conjunto de entrenamiento.

Hay que tener en cuenta que los autores de los datos proporcionan unos datos de validación y entrenamiento. En estos splits los datos de entrenamiento tienen 100 clases y los de validación 50, emulando las clases unseen. Pero eso a nosotros no nos interesa porque lo que queremos es hacer un cross-validation. En el que todas las clases hayan sido unseen al menos una vez en cada fold

In [3]:
SHOW = False 

# Set up the path to the data
path      = pathlib.Path('data/')

path_cub  = path / 'CUB/'
path_awa2 = path / 'AWA2/'
path_FLO  = path / 'FLO/'
path_sun  = path / 'SUN/'


# Load the data
matcontent, att_splits = load_data_from_path(path_cub, show = False )

"""
Comentar que en los datos de este datasets viene una partición establecida para train y val. Sin embargo, la partición de 'val' no contempla clases Unseen. Por tanto no la vamos usar.
Crearemos una partición de validación para clases Seen y Unseen a partir de la partición de test. 
Como conjunto de entrenamiento usaremos el que viene dado por los índices de la variable 'trainval_loc' en att_splits. 
Los índices de las varaibles 'train_loc' y 'val_loc' no los vamos a usar.
"""
# Get the data from att_splits dictionary
allclasses_names            = att_splits['allclasses_names']
original_att                = att_splits['original_att'].T.astype('float')  # Attributos originales
attribute                   = att_splits['att'].T.astype('float')           # Atributos que no se de donde salen, pero son los que usan <=== IMPORTANTE
# Indixes of the instances in the train, test and validation sets
test_unseen_loc             = att_splits['test_unseen_loc'].squeeze() - 1
test_seen_loc               = att_splits['test_seen_loc'].squeeze() - 1
trainval_loc                = att_splits['trainval_loc'].squeeze() - 1      # En caso de no usar validación, aquí estara todo el entrenamiento
train_loc                   = att_splits['train_loc'].squeeze() - 1         # Esto es si usamos validacion, ver comentario arriba
val_loc                     = att_splits['val_loc'].squeeze() -1            # Esto es si usamos validacion, ver comentario arriba
# Get the dta from matcontent dictionary
feature =  matcontent['features'].T                       # Feature Matrix (Number of instances x Number of features)
labels  =  matcontent['labels'].astype(int).squeeze() - 1 # Ponemos que sea entero, eliminamos la dimension extra y con el '-1' movemos el rango de [1,200] a [0,199]



In [4]:
seed           = 42
number_folds   = 5
scaler_str     = 'Standard'     # ['Standard','MinMax']
orig_attribute = False # Attributos originales u otros
random.seed(seed)

random.shuffle(trainval_loc)
trainval_feature = feature[trainval_loc]
trainval_labels  = labels[trainval_loc]
trainval_classes = np.unique(trainval_labels)
random.shuffle(trainval_classes)

classes_per_fold = int(trainval_classes.shape[0] / number_folds)
acarreo          = trainval_classes.shape[0] % number_folds

# Get attributes and preprocessing them
attribute = original_att if orig_attribute else attribute
attribute = scaler_data(get_scaler(scaler_str),attribute)

folds = []
for i in range(number_folds):
    
    # Gather data for the fold
    start = i * classes_per_fold
    end   = (i + 1) * classes_per_fold if i != number_folds - 1 else (i + 1) * classes_per_fold + acarreo
    
    fold_i_val_classes       = trainval_classes[start:end] # Unseen
    fold_i_training_classes  = [x for x in trainval_classes if x not in fold_i_val_classes] # Seen
    
    fold_i_val_loc           = np.where(np.isin(trainval_labels, fold_i_val_classes)) # index for val
    fold_i_training_loc      = np.where(np.isin(trainval_labels, fold_i_training_classes)) # Index for training

    assert np.intersect1d(fold_i_val_classes, fold_i_training_classes).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert all([i not in fold_i_training_classes for i in trainval_labels[fold_i_val_loc]]), f"Error: Problems found in creating folds {i}"
    assert all([i not in fold_i_val_classes for i in trainval_labels[fold_i_training_loc]]), f"Error: Problems found in creating folds {i}"

    fold_i_training_features = feature[fold_i_training_loc]
    fold_i_val_features      = feature[fold_i_val_loc]
    fold_i_training_labels   = labels[fold_i_training_loc]
    fold_i_val_labels        = labels[fold_i_val_loc]
    fold_i_ss_val            = gen_ss_from_data(fold_i_val_labels,attribute)
    fold_i_ss_training       = gen_ss_from_data(fold_i_training_labels,attribute)
    
    # Preprocess the data
    fold_i_training_features, fold_i_val_features = scaler_data(get_scaler(scaler_str),
                                                                fold_i_training_features,
                                                                fold_i_val_features)
    
    # Creting Datasets 
    fold_training = Dataset(features = fold_i_training_features, 
                            labels   = fold_i_training_labels, 
                            att      = fold_i_ss_training, 
                            mode     = 'train')
    
    fold_val      = Dataset(features = fold_i_val_features,
                            labels   = fold_i_val_labels,
                            att      = fold_i_ss_val,
                            mode     = 'val')
    
    fold = StackedDataset({'train':fold_training, 'val':fold_val})
    
    folds.append(fold)

In [26]:
step = 5
debug = True
results  = []
mascaras = {}
hitk = 1    


vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[0] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])
    
    
# Tomamos el primer fold com training data
training_data = folds[0].datasets['train']
val_data      = folds[0].datasets['val']

attributes_X_train = training_data.att.copy()
attributes_X_val   = val_data.att.copy()

rfe = RFE(SVC(kernel='linear'), n_features_to_select=1)
rfe.fit(attributes_X_train, training_data.labels)
ranking = rfe.ranking_


In [43]:
from SAE import *
from auto_tqdm import tqdm

In [45]:
DECIMAL_NUMBERS = 3
debug           = True
conteo          = np.zeros(attribute.shape[1])

best_folds_configuration = []

for i, fold in tqdm(enumerate(folds)):

    
    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']

    attributes_X_train = training_data.att.copy()


    rfe = RFE(SVC(kernel='linear'), n_features_to_select=1)
    rfe.fit(attributes_X_train, training_data.labels)
    ranking = rfe.ranking_
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <=nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Evaluo en trainig
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Evaluo en val
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # guardo los resultados
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Ordeno los resultados en función del mejor en validación
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    # Obtengo la mejor configuración para cada fold
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

0it [00:00, ?it/s]

nv: 100 || Train: 95.563 || Val: 61.0
nv: 175 || Train: 96.376 || Val: 62.714
nv: 240 || Train: 96.5 || Val: 62.071
nv: 310 || Train: 96.535 || Val: 63.071
nv: 100 || Train: 95.708 || Val: 63.413
nv: 175 || Train: 96.406 || Val: 64.164
nv: 240 || Train: 96.638 || Val: 63.686
nv: 310 || Train: 96.62 || Val: 64.369
nv: 100 || Train: 95.786 || Val: 61.977
nv: 175 || Train: 96.456 || Val: 62.915
nv: 240 || Train: 96.367 || Val: 63.781
nv: 310 || Train: 96.35 || Val: 62.987
nv: 100 || Train: 95.753 || Val: 62.66
nv: 175 || Train: 96.549 || Val: 65.22
nv: 240 || Train: 96.496 || Val: 66.216
nv: 310 || Train: 96.532 || Val: 65.861
nv: 100 || Train: 95.669 || Val: 60.643
nv: 175 || Train: 96.641 || Val: 63.143
nv: 240 || Train: 96.111 || Val: 62.286
nv: 310 || Train: 96.288 || Val: 62.5


In [51]:
zeros = np.zeros(attribute.shape[1])

for res in best_folds_configuration:
    zeros = zeros + np.asarray(res['Mask']).astype(int)
    
zeros

array([5., 5., 5., 2., 5., 5., 5., 5., 5., 4., 4., 4., 4., 2., 5., 2., 2.,
       2., 5., 5., 5., 4., 4., 5., 5., 4., 4., 4., 5., 5., 2., 2., 2., 4.,
       5., 5., 5., 2., 5., 2., 5., 2., 5., 5., 5., 5., 2., 2., 5., 5., 5.,
       5., 5., 5., 5., 4., 4., 5., 2., 4., 2., 2., 2., 5., 2., 2., 2., 5.,
       4., 5., 4., 2., 5., 4., 5., 5., 5., 5., 5., 5., 4., 4., 5., 2., 5.,
       2., 4., 2., 4., 4., 5., 5., 0., 5., 5., 5., 5., 2., 5., 2., 5., 5.,
       5., 5., 4., 2., 5., 2., 5., 4., 5., 4., 4., 2., 5., 5., 5., 5., 5.,
       4., 5., 5., 2., 5., 2., 5., 4., 4., 2., 2., 2., 5., 5., 2., 5., 4.,
       5., 5., 5., 5., 4., 5., 5., 5., 2., 5., 5., 5., 5., 5., 5., 5., 4.,
       4., 2., 4., 2., 5., 2., 5., 2., 2., 4., 5., 5., 2., 5., 5., 4., 4.,
       5., 2., 5., 2., 2., 2., 5., 4., 5., 5., 2., 5., 2., 4., 2., 2., 2.,
       5., 2., 5., 2., 4., 5., 5., 5., 2., 4., 4., 5., 2., 4., 4., 5., 5.,
       2., 0., 5., 5., 5., 5., 4., 5., 5., 5., 5., 5., 5., 5., 5., 5., 5.,
       4., 5., 5., 4., 5.

In [61]:
np.where(zeros >= 1)[0].shape

(310,)

In [46]:
best_folds_configuration

[Attributes                                                  310
 Train Acc                                                96.535
 Val Acc                                                  63.071
 Mask          [True, True, True, True, True, True, True, Tru...
 Name: 3, dtype: object,
 Attributes                                                  310
 Train Acc                                                 96.62
 Val Acc                                                  64.369
 Mask          [True, True, True, True, True, True, True, Tru...
 Name: 3, dtype: object,
 Attributes                                                  240
 Train Acc                                                96.367
 Val Acc                                                  63.781
 Mask          [True, True, True, False, True, True, True, Tr...
 Name: 2, dtype: object,
 Attributes                                                  240
 Train Acc                                                96.496
 Val Acc       

In [37]:
acc = []
DECIMAL_NUMBERS = 3
for nv in [175,245,310]:
    mask = ranking <= nv
    
    W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
    
    gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
    semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
    [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
    zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
    
    
    gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
    semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
    [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
    zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
    
    print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")

nv: 175 || Train: 96.376 || Val: 62.714
nv: 245 || Train: 96.429 || Val: 61.786
nv: 310 || Train: 96.535 || Val: 63.071


In [32]:
training_data.att[:,mask].shape

(5657, 10)

In [38]:
mask = ranking <= 10
mask.sum()

10

# Anexos

Aquí debajo intento poner la partición en modo de función-generador

In [65]:
seed           = 42
number_folds   = 5
scaler_str     = 'Standard'     # ['Standard','MinMax']
orig_attribute = False # Attributos originales u otros
random.seed(seed)

# Get attributes and preprocessing them
attribute = original_att if orig_attribute else attribute
attribute = scaler_data(get_scaler(scaler_str),attribute)

random.shuffle(trainval_loc)
trainval_feature = feature[trainval_loc]
trainval_labels  = labels[trainval_loc]
trainval_classes = np.unique(trainval_labels)
random.shuffle(trainval_classes)

def split_into_folds(trainval_feature,trainval_labels,number_folds: int):

    classes_per_fold = int(trainval_classes.shape[0] / number_folds)
    acarreo          = trainval_classes.shape[0] % number_folds

    folds = []
    for i in range(number_folds):
        
        # Gather data for the fold
        start = i * classes_per_fold
        end   = (i + 1) * classes_per_fold if i != number_folds - 1 else (i + 1) * classes_per_fold + acarreo
        
        fold_i_val_classes       = trainval_classes[start:end] # Unseen
        fold_i_training_classes  = [x for x in trainval_classes if x not in fold_i_val_classes] # Seen
        
        fold_i_val_loc           = np.where(np.isin(trainval_labels, fold_i_val_classes)) # index for val
        fold_i_training_loc      = np.where(np.isin(trainval_labels, fold_i_training_classes)) # Index for training

        assert np.intersect1d(fold_i_val_classes, fold_i_training_classes).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
        assert all([i not in fold_i_training_classes for i in trainval_labels[fold_i_val_loc]]), f"Error: Problems found in creating folds {i}"
        assert all([i not in fold_i_val_classes for i in trainval_labels[fold_i_training_loc]]), f"Error: Problems found in creating folds {i}"

        fold_i_training_features = trainval_feature[fold_i_training_loc]
        fold_i_val_features      = trainval_feature[fold_i_val_loc]
        fold_i_training_labels   = trainval_labels[fold_i_training_loc]
        fold_i_val_labels        = trainval_labels[fold_i_val_loc]
        fold_i_ss_val            = gen_ss_from_data(fold_i_val_labels,attribute)
        fold_i_ss_training       = gen_ss_from_data(fold_i_training_labels,attribute)
        
        # Preprocess the data
        fold_i_training_features, fold_i_val_features = scaler_data(get_scaler(scaler_str),
                                                                    fold_i_training_features,
                                                                    fold_i_val_features)
        
        # Creting Datasets 
        fold_training = Dataset(features = fold_i_training_features, 
                                labels   = fold_i_training_labels, 
                                att      = fold_i_ss_training, 
                                mode     = 'train')
        
        fold_val      = Dataset(features = fold_i_val_features,
                                labels   = fold_i_val_labels,
                                att      = fold_i_ss_val,
                                mode     = 'val')
        
        fold = StackedDataset({'train':fold_training, 'val':fold_val})
        
        yield fold

In [68]:
trainval_labels.shape

(7057,)

# Lectura general de los datos